In [1]:
!poetry install -q

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
import datetime
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# OpenMP 다중 로드 허용 및 스레드 경쟁 방지 환경 변수 (최상단 주입 필수)
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

In [3]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [4]:
from pathlib import Path
from typing import Any, Dict, Optional
import pandas as pd

from src.test.backtest.backtest_loader import (
    LoadedBacktestArtifacts,
    load_backtest_artifacts,
)

# ==============================================================================
# 0. 백테스트 아티팩트 일괄 로드 (무결성 컨테이너 획득)
# ==============================================================================
# [설계 의도] auto_display=True를 통해 실행 메타, 성과 감사표, 노출도 통계 및 무결성 테이블 자동 사출
loaded_backtest: LoadedBacktestArtifacts = load_backtest_artifacts(
    artifact_root="artifacts/backtest",
    auto_display=False,
)

# ==============================================================================
# 1. 설정 및 매니페스트 메타데이터 계층 (Governance Config & Manifest)
# ==============================================================================
# [설계 의도] 백테스트 실행 환경, 평가 기간 및 동결된 8대 팩터 XAI 거버넌스 파라미터(θ, σ, w, s) 바인딩
backtest_manifest: Dict[str, Any] = loaded_backtest.manifest
governance_config: Dict[str, Any] = loaded_backtest.governance_config

# ==============================================================================
# 2. 시계열 원천 및 시뮬레이션 데이터 계층 (Gates, Signals, Ledgers)
# ==============================================================================
# [설계 의도] 일별 복합 리스크 게이트(G_t), 개별 침범 거리, 전략별 주문 액션 및 3대 전략 회계 원장 바인딩
xai_risk_gates: pd.DataFrame = loaded_backtest.xai_risk_gates
strategy_signals: pd.DataFrame = loaded_backtest.strategy_signals
simulation_ledgers: pd.DataFrame = loaded_backtest.simulation_ledgers

# ==============================================================================
# 3. 정량 성과 및 거버넌스 감사 보고서 계층 (Quantitative Reports)
# ==============================================================================
# [설계 의도] ABtest 통계 검정 및 전략 비교 분석에 즉시 투입 가능한 정량 감사표 6종 바인딩
ablation_performance_metrics: pd.DataFrame = loaded_backtest.ablation_performance_metrics
execution_governance_audit: pd.DataFrame = loaded_backtest.execution_governance_audit
portfolio_accounting_summary: pd.DataFrame = loaded_backtest.portfolio_accounting_summary
gate_summary: pd.DataFrame = loaded_backtest.gate_summary
regime_attribution: pd.DataFrame = loaded_backtest.regime_attribution
factor_audit_trail: pd.DataFrame = loaded_backtest.factor_audit_trail

# ==============================================================================
# 4. 아티팩트 물리 경로 매핑 (Artifact File Paths)
# ==============================================================================
# [설계 의도] 고해상도 진단 플롯(PNG) 및 원천 Parquet/CSV/JSON 파일의 물리적 경로 매핑 바인딩
backtest_artifact_paths: Dict[str, Path] = loaded_backtest.artifact_paths

In [5]:
TOTAL_TRADING_DAYS: int = backtest_manifest["evaluation_period"]["trading_days"]
CV_TRADING_DAYS: int = backtest_manifest["evaluation_period"]["cv_trading_days"]
HOLDING_PERIOD: int = backtest_manifest["evaluation_period"]["holding_period"]

In [6]:
# ==============================================================================
# [셀 1] 사전 실험계획 및 통계적 검정력 분석 (Ex-ante Power Analysis)
# ==============================================================================
from src.test.ab_test.power_analysis import run_ex_ante_power_analysis

# Cohen's d >= 0.25, 유의수준 α = 0.05 하에서 목표 검정력(1 - β >= 0.80) 충족 여부를 사전 감사합니다.
power_audit_result = run_ex_ante_power_analysis(
    loaded_artifacts=loaded_backtest,
    total_trading_days=TOTAL_TRADING_DAYS,
    holding_period=HOLDING_PERIOD,
    cv_trading_days=CV_TRADING_DAYS,
)

power_audit_table: pd.DataFrame = (
    power_audit_result["power_audit_table"]
    if isinstance(power_audit_result, dict) and "power_audit_table" in power_audit_result
    else power_audit_result
)

print("=" * 100)
print("사전 통계적 검정력 및 유효 표본 크기 감사 판정표 (Ex-ante Power Audit Table)")
print("=" * 100)
display(power_audit_table)

사전 통계적 검정력 및 유효 표본 크기 감사 판정표 (Ex-ante Power Audit Table)


,Evaluation Scope,Observed Days,Holding Period,Effective N (N_eff),Target MDE (Cohen's d),Achievable MDE,Achieved Power (1-β),Target Power,Required N_eff,Required Days,Audit Verdict
0,Test Horizon (Overall 509d),509,20,25,0.25,0.5119,0.3336,0.8,101,2020,WARNING (Underpowered)
1,CV Window (Frozen 203d),203,20,10,0.25,0.8528,0.1807,0.8,101,2020,WARNING (Underpowered)


In [7]:
from src.test.ab_test.experiment_setup import (
    ExperimentContext,
    setup_experiment_environment,
)

# ==============================================================================
# [Cell 3 / Pre-setup] 기개발 산출물 바인딩 및 평가 관측창 무결성 동결
# ==============================================================================
# [설계 의도] 셀 1에서 사전 로드된 loaded_backtest 컨테이너를 주입하여 509 거래일 시계열 정렬,
# 결측치(0 NaN) 및 원시 대조군(Raw) vs 방어형 거버넌스(XAI-Gated) 벡터 바인딩 무결성을 검증합니다.
experiment_context: ExperimentContext = setup_experiment_environment(
    loaded_artifacts=loaded_backtest,
    total_trading_days=TOTAL_TRADING_DAYS,
    holding_period=HOLDING_PERIOD,
    cv_trading_days=CV_TRADING_DAYS,
)

print("=" * 100)
print("[Pre-setup] 실험 환경 동결 및 시계열 정렬 무결성 검증 요약표 (Integrity Summary Table)")
print("=" * 100)
display(experiment_context.summary_table)

[Pre-setup] 실험 환경 동결 및 시계열 정렬 무결성 검증 요약표 (Integrity Summary Table)


,Audit Metric,Observed Value,Reference Baseline,Integrity Status
0,Evaluation Period,2024-06-19 ~ 2026-07-23,Test Horizon (Target 509d),VERIFIED
1,Total Trading Days (T),509 days,509 days,MATCH
2,Effective Sample Size (N_eff),25 blocks,>= 10 blocks (CV Frozen),POWER_SATISFIED
3,Series Alignment & NaN Count,0 NaNs,0 NaNs Required,CLEAN
4,Target Comparison Group,Strategy_1_Raw_Model vs Strategy_2_XAI-Gated,Axis 3 (Raw Baseline vs Risk-Gated),BOUND


In [8]:
from typing import Any, Dict

from src.test.ab_test.axis_governance import evaluate_defense_governance

# ==============================================================================
# [Cell 4 / Phase 3] Axis 3: XAI 방어 거버넌스 개입 유효성 단측 검정 및 TOST 동등성 검정
# ==============================================================================
# [설계 의도] Shapiro-Wilk 정규성 진단에 따라 Nadeau-Bengio 대응표본 t-검정 또는 Pratt 부호순위검정으로
# 동적 라우팅하고, Holm-Bonferroni(제1종 오류 통제) 및 비개입 평온장 TOST(제2종 오류 통제)를 집행합니다.
governance_evaluation: Dict[str, Any] = evaluate_defense_governance(
    simulation_ledgers=simulation_ledgers,
    strategy_signals=strategy_signals,
)

print("=" * 100)
print("[Axis 3 - Table 1] 하방 리스크 억제 단측 가설 검정 및 제1·2종 오류 감사표 (Hypothesis Audit)")
print("=" * 100)
display(governance_evaluation["hypothesis_audit_table"])

print("\n" + "=" * 100)
print("[Axis 3 - Table 2] 비개입 평온장(G_t >= 0.99) 원시 성능 보존 TOST 동등성 감사표 (Equivalence Audit)")
print("=" * 100)
display(governance_evaluation["equivalence_audit_table"])

print("\n" + "=" * 100)
print("[Axis 3 - Table 3] 1차 핵심 지표 및 2차 가드레일 지표 비교 요약표 (Guardrail Summary)")
print("=" * 100)
display(governance_evaluation["guardrail_summary_table"])

[Axis 3 - Table 1] 하방 리스크 억제 단측 가설 검정 및 제1·2종 오류 감사표 (Hypothesis Audit)


,Metric,Sample Size (N),Normality (Shapiro p),Routed Test,Weight (w),Mean Diff (Risk - Raw),Bootstrap 95% CI,Effect Size (dz),Statistic,Raw p-value,Adj p-value,Post-hoc Power (1-β),Type II Error (β),Reject H0 (α=0.05)
Hypothesis,,,,,,,,,,,,,,
H1_Risk_MDD_Suppression,Rolling Maximum Drawdown |MDD| (60d),509,0.0,Signed-Rank (pratt) + CBB(b=20),0.37,-0.018574,"[-0.030071, -0.009558]",-0.6088,-4.0046,0.000031,0.000083,0.9052,0.0948,True
H1_Risk_Sortino_Improvement,Rolling Sortino Ratio (60d),509,0.0,Signed-Rank (pratt) + CBB(b=20),0.37,0.533102,"[0.233934, 0.843708]",0.5366,2.5524,0.005349,0.009048,0.8318,0.1682,True
H1_Risk_Intervention_Drawdown_Suppression,Intervention |Drawdown| (G_t < 0.8000),235,0.0,Signed-Rank (pratt) + CBB(b=20),0.26,-0.017924,"[-0.034094, -0.005076]",-0.5170,-2.0678,0.019331,0.019331,0.4811,0.5189,True



[Axis 3 - Table 2] 비개입 평온장(G_t >= 0.99) 원시 성능 보존 TOST 동등성 감사표 (Equivalence Audit)


,Calm Days (N),Equivalence Margin (±δ),Mean Return Diff,Normality (Shapiro p),Routed Test,Lower Bound p-val (> -δ),Upper Bound p-val (< +δ),TOST Combined p-val,Equivalence Established
Regime,,,,,,,,,
Non-Intervention Calm (G_t >= 0.80),274,±0.0005,-0.000041,0.0,TOST Signed-Rank (pratt),0.001388,0.001071,0.001388,True



[Axis 3 - Table 3] 1차 핵심 지표 및 2차 가드레일 지표 비교 요약표 (Guardrail Summary)


Strategy 1 (Raw Model)  Strategy 2 (XAI-Gated)  Delta (Risk - Raw)
Metric Category                      Metric Name                                                                                   
Primary (Downside)                   Maximum Drawdown (%)                    -22.110000              -14.730000            7.380000
                                     Sortino Ratio                             2.225000                2.938000            0.713000
Primary (Risk-Adj)                   Annualized Sharpe Ratio                   1.489000                1.818000            0.329000
Guardrail (HAC Significance)         Newey-West t-statistic                    2.248000                2.478000            0.230000
                                     Newey-West p-value                        0.034017                0.020623           -0.013394
Guardrail (Calm Regime Preservation) Calm Regime Drag Ratio (%)                0.000000                2.466500            2.466500